# nb_silver_build

Flattens Bronze JSON files to Silver Delta tables in lh_health_analytics.

See SILVER_NOTEBOOK.md for walkthrough.

In [ ]:
# Cell 1 — Imports, silver schema, BOM-safe reader
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, LongType, ArrayType
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# Bronze files have a UTF-8 BOM — strip it before Spark parses JSON
def read_json(path, schema=None):
    rdd = spark.sparkContext.wholeTextFiles(path).values()\
               .map(lambda x: x.lstrip('\ufeff'))
    if schema:
        return spark.read.schema(schema).json(rdd)
    return spark.read.json(rdd)

print("Setup complete")
# Result: Setup complete

In [ ]:
# Cell 2 — silver.datasets
BRONZE = "Files/bronze"

ds_raw = read_json(f"{BRONZE}/datasets.json")
ds = ds_raw.select(F.explode("result").alias("r")).select("r.*")
ds.write.format("delta").mode("overwrite").saveAsTable("silver.datasets")
print(f"silver.datasets: {ds.count()} rows")
# Result: 7199 rows

In [ ]:
# Cell 3 — silver.measures
ms_raw = read_json(f"{BRONZE}/measures.json")
ms = ms_raw.select(F.explode("result").alias("r")).select("r.*")
ms.write.format("delta").mode("overwrite").saveAsTable("silver.measures")
print(f"silver.measures: {ms.count()} rows")
# Result: 33 rows

In [ ]:
# Cell 4 — silver.measure_categories
mc_raw = read_json(f"{BRONZE}/measure_categories.json")
mc = mc_raw.select(F.explode("result").alias("r")).select("r.*")
mc.write.format("delta").mode("overwrite").saveAsTable("silver.measure_categories")
print(f"silver.measure_categories: {mc.count()} rows")
# Result: 12 rows

In [ ]:
# Cell 5 — silver.reporting_unit_types + silver.reporting_units
rut_raw = read_json(f"{BRONZE}/reporting_unit_types.json")
rut = rut_raw.select(F.explode("result").alias("r")).select("r.*")
rut.write.format("delta").mode("overwrite").saveAsTable("silver.reporting_unit_types")
print(f"silver.reporting_unit_types: {rut.count()} rows")
# Result: 6 rows

ru_raw = read_json(f"{BRONZE}/reporting_units.json")
ru = ru_raw.select(F.explode("result").alias("r")).select("r.*")
ru.write.format("delta").mode("overwrite").saveAsTable("silver.reporting_units")
print(f"silver.reporting_units: {ru.count()} rows")
# Result: 1427 rows

In [ ]:
# Cell 6 — silver.data_items (explicit schema — 33 files, ~1.2 GB)
item_schema = StructType([
    StructField("result", ArrayType(StructType([
        StructField("caveats", ArrayType(StringType()), True),
        StructField("data_set_id", IntegerType(), True),
        StructField("group_number", StringType(), True),
        StructField("lower_value", DoubleType(), True),
        StructField("measure_code", StringType(), True),
        StructField("peer_group_summary", StringType(), True),
        StructField("proxy_reporting_unit_summary", StringType(), True),
        StructField("reported_measure_code", StringType(), True),
        StructField("reporting_unit_summary", StructType([
            StructField("reporting_unit_code", StringType(), True),
            StructField("reporting_unit_name", StringType(), True),
            StructField("reporting_unit_type", StructType([
                StructField("reporting_unit_type_code", StringType(), True),
                StructField("reporting_unit_type_name", StringType(), True)
            ]), True)
        ]), True),
        StructField("suppressions", ArrayType(StringType()), True),
        StructField("upper_value", DoubleType(), True),
        StructField("value", DoubleType(), True)
    ])), True)
])

di_raw = read_json(f"{BRONZE}/data_items", schema=item_schema)
di = (di_raw
      .select(F.explode("result").alias("r"))
      .select(
          F.col("r.data_set_id"),
          F.col("r.measure_code"),
          F.col("r.reported_measure_code"),
          F.col("r.reporting_unit_summary.reporting_unit_code").alias("reporting_unit_code"),
          F.col("r.reporting_unit_summary.reporting_unit_name").alias("reporting_unit_name"),
          F.col("r.reporting_unit_summary.reporting_unit_type.reporting_unit_type_code").alias("reporting_unit_type_code"),
          F.col("r.value"),
          F.col("r.lower_value"),
          F.col("r.upper_value"),
          F.col("r.group_number"),
          F.size(F.col("r.suppressions")).alias("suppression_count"),
          F.size(F.col("r.caveats")).alias("caveat_count")
      ))
di.write.format("delta").mode("overwrite").saveAsTable("silver.data_items")
print(f"silver.data_items: {di.count()} rows")
# Result: 1700870 rows